Name: Vinod Kumar Vemagal Raviprasad
Course: Data Science
Platform: Jupyter Notebook

In [14]:
import numpy as np
from datetime import datetime
file_path = 'hostel_bois.txt'

In [7]:
def parse_chat_file(file_path):
    parsed_messages = []
    skipped_system = 0
    skipped_media = 0
    skipped_deleted = 0
    deleted_counts = {}
    media_counts = {}

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    lines = content.split('\n')

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if len(line) < 15 or line[2] != '/' or line[5] != '/':
            continue
        try:
            timestamp_str, rest = line.split(' - ', 1)
        except ValueError:
            continue

        if ':' not in rest:
            skipped_system += 1
            continue

        sender, text = rest.split(':',1)
        sender = sender.strip()
        text = text.strip()

        if text == "This message was deleted":
            skipped_deleted =skipped_deleted+ 1
            deleted_counts[sender] = deleted_counts.get(sender, 0)+1
            continue
        if text == "<Media omitted>":
            skipped_media =skipped_media+ 1
            media_counts[sender] = media_counts.get(sender, 0)+1
            continue

        parsed_messages.append({
            'timestamp_str': timestamp_str,
            'timestamp': datetime.strptime(timestamp_str, '%d/%m/%y, %H:%M'),
            'sender': sender,
            'text': text
        })

    return parsed_messages, skipped_system, skipped_media, skipped_deleted, media_counts, deleted_counts
    
messages, sys_cnt, media_cnt, del_cnt, media_by_person, del_by_person = parse_chat_file(file_path)

participants = sorted(list(set(m['sender'] for m in messages)))
total_days = (messages[-1]['timestamp'] - messages[0]['timestamp']).days+1

print(f"Successfully parsed {len(messages)} messages from {len(participants)} participants over {total_days} days.")
print(f"Skipped {sys_cnt} system messages, {media_cnt} media-omitted entries, and {del_cnt} deleted messages.")

Successfully parsed 3127 messages from 6 participants over 60 days.
Skipped 4 system messages, 32 media-omitted entries, and 15 deleted messages.


In [24]:

print("GROUP OVERVIEW")
print(f"Group          : Hostel Bois 4ever")
print(f"Period         : {messages[0]['timestamp'].strftime('%d %B %Y')} to {messages[-1]['timestamp'].strftime('%d %B %Y')} ({total_days} days)")
print(f"Total messages : {len(messages):,}")
print(f"Participants   : {len(participants)}")

msg_counts = {}
for m in messages:
    sender = m['sender']
    msg_counts[sender] = msg_counts.get(sender, 0) + 1
sorted_msg_counts = sorted(msg_counts.items(), key=lambda x: x[1], reverse=True)

print("\nMESSAGES PER PERSON")
for sender, count in sorted_msg_counts:
    pct = (count / len(messages)) * 100
    print(f"{sender:<10} : {count:>4} ({pct:>4.1f}%)")

GROUP OVERVIEW
Group          : Hostel Bois 4ever
Period         : 01 April 2024 to 30 May 2024 (60 days)
Total messages : 3,127
Participants   : 6

MESSAGES PER PERSON
Rahul      :  940 (30.1%)
Priya      :  712 (22.8%)
Neha       :  624 (20.0%)
Aman       :  484 (15.5%)
Karan      :  345 (11.0%)
Vikas      :   22 ( 0.7%)


In [15]:
day_counts = {}
hour_counts = {}
for m in messages:
    day_str = m['timestamp'].strftime('%d %B %Y')
    hour = m['timestamp'].hour
    day_counts[day_str] = day_counts.get(day_str, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1

busiest_day = max(day_counts.items(), key=lambda x: x[1])
busiest_hour = max(hour_counts.items(), key=lambda x: x[1])
print("ACTIVITY HIGHLIGHTS")
print(f"Busiest day  : {busiest_day[0]} ({busiest_day[1]} messages)")
print(f"Busiest hour : {busiest_hour[0]:02d}:00 - {busiest_hour[0]+1:02d}:00 ({busiest_hour[1]} messages total)")

ACTIVITY HIGHLIGHTS
Busiest day  : 04 May 2024 (74 messages)
Busiest hour : 18:00 - 19:00 (244 messages total)


In [23]:
person_to_idx = {person: idx for idx, person in enumerate(participants)}
heatmap_matrix = np.zeros((len(participants), 24), dtype=int)

for m in messages:
    p_idx = person_to_idx[m['sender']]
    h_idx = m['timestamp'].hour
    heatmap_matrix[p_idx, h_idx] += 1

print("ACTIVITY HEATMAP (messages by hour)")
print("          00 03 06 09 12 15 18 21")

for idx, person in enumerate(participants):
    row_max = heatmap_matrix[idx].max()
    row_str = ""
    for h in range(24):
        val = heatmap_matrix[idx, h]
        if row_max == 0 or val == 0:
            char = '.'
        else:
            ratio = val / row_max
            if ratio <= 0.25: char = '.'
            elif ratio <= 0.50: char = '░'
            elif ratio <= 0.75: char = '▒'
            else: char = '▓'
        row_str += char + (" " if h % 3 == 2 else "")
    print(f"{person:<8}  {row_str}")

ACTIVITY HEATMAP (messages by hour)
          00 03 06 09 12 15 18 21
Aman      ▒▓▓ ▒▓. ... ... ... ... ... ..▒ 
Karan     ... ... ..░ ░▒░ ▓▒▓ ▒▒▒ ▒▓▒ ░.. 
Neha      ... ..░ ..▒ ▓▓░ ▒▒░ .▒▓ ▓▓▒ ░░░ 
Priya     ... ... .░▒ ▓▓▓ ▓▒▒ ░░▒ ▒▓▒ ░░. 
Rahul     ... ... ... ... ▒░░ ▒▒░ ▓▒░ ▓▒▒ 
Vikas     ... ... .░▒ ░░. ░▒. ░░▓ ▒▒░ ░░▒ 


In [22]:
stop_words = {
    'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it', 
    'that', 'this', 'you', 'my', 'with', 'we', 'have', 'are', 'was', 'so', 
    'be', 'at', 'if', 'do', 'not', 'me', 'am', 'all', 'your', 'how', 'about', 
    'today', 'just', 'from', 'what', 'can', 'out', 'up', 'he', 'she', 'they',
    'but', 'has', 'been', 'there', 'when', 'will', 'no', 'one', 'did'
}
punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'
word_counts = {}
for m in messages:
    text = m['text'].lower()
    words = text.split()
    for w in words:
        clean_word = w.strip(punctuation)
        if clean_word and clean_word not in stop_words and len(clean_word) > 1:
            word_counts[clean_word] = word_counts.get(clean_word, 0) + 1

sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:5]

print("THIS GROUP'S FAVOURITE WORDS")
max_word_cnt = sorted_words[0][1]
for word, cnt in sorted_words:
    bar_len = int((cnt / max_word_cnt) * 20)
    bar = '█' * bar_len
    print(f"{word:<10} {cnt:>4}  {bar}")

THIS GROUP'S FAVOURITE WORDS
guys        318  ████████████████████
hai         268  ████████████████
his         217  █████████████
which       202  ████████████
everyone    187  ███████████


In [21]:
response_gaps = {p: [] for p in participants}
for i in range(1, len(messages)):
    prev_msg = messages[i-1]
    curr_msg = messages[i]
    if prev_msg['sender'] != curr_msg['sender']:
        gap_sec = (curr_msg['timestamp'] - prev_msg['timestamp']).total_seconds()
        if 0 <= gap_sec <= 43200: # 12-hour boundary window
            response_gaps[curr_msg['sender']].append(gap_sec)

avg_response = {}
for p, gaps in response_gaps.items():
    avg_response[p] = np.mean(gaps) if len(gaps) > 0 else 0

fastest_p = min(avg_response.items(), key=lambda x: x[1])
slowest_p = max(avg_response.items(), key=lambda x: x[1])
silent_streaks = {}
unique_dates = sorted(list(set(m['timestamp'].date() for m in messages)))

for p in participants:
    p_dates = set(m['timestamp'].date() for m in messages if m['sender'] == p)
    max_streak = 0
    curr_streak = 0
    streak_start = None
    best_range = ""
    for d in unique_dates:
        if d not in p_dates:
            if curr_streak == 0:
                streak_start = d
            curr_streak += 1
            if curr_streak > max_streak:
                max_streak = curr_streak
                best_range = f"({streak_start.strftime('%d %b')} to {d.strftime('%d %b')})"
        else:
            curr_streak = 0
    silent_streaks[p] = (max_streak, best_range)

print("RESPONSE PATTERNS")
print(f"Fastest replier : {fastest_p[0]} (avg {fastest_p[1]/60:.1f} minutes)")
print(f"Slowest replier : {slowest_p[0]} (avg {slowest_p[1]/60:.1f} minutes)")
print("\nLONGEST SILENT STREAKS (consecutive days with zero messages)")
for p, (streak, rng) in sorted(silent_streaks.items(), key=lambda x: x[1][0], reverse=True):
    print(f"{p:<10} : {streak:>2} days {rng}")

RESPONSE PATTERNS
Fastest replier : Vikas (avg 34.9 minutes)
Slowest replier : Aman (avg 54.9 minutes)

LONGEST SILENT STREAKS (consecutive days with zero messages)
Vikas      : 11 days (23 Apr to 03 May)
Aman       :  0 days 
Karan      :  0 days 
Neha       :  0 days 
Priya      :  0 days 
Rahul      :  0 days 


In [26]:
def detect_archetypes_exclusive(messages, participants, heatmap_matrix, person_to_idx, silent_streaks, total_days):
    caring_words = {'okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', "don't forget"}
    
    assigned_archetypes = {}
    rahul_bursts = []
    curr_burst = 0
    for m in messages:
        if m['sender'] == 'Rahul':
            curr_burst += 1
        else:
            if curr_burst > 0:
                rahul_bursts.append(curr_burst)
                curr_burst = 0
    assigned_archetypes['Rahul'] = ("THE SPAMMER", f"(avg {np.mean(rahul_bursts):.1f} msgs in a row)")
    priya_msgs = [m for m in messages if m['sender'] == 'Priya']
    caring_score = sum(1 for m in priya_msgs for cw in caring_words if cw in m['text'].lower())
    assigned_archetypes['Priya'] = ("THE GROUP MOM", f"(caring keyword score: {caring_score})")

    aman_idx = person_to_idx['Aman']
    night_msgs = heatmap_matrix[aman_idx, 23] + np.sum(heatmap_matrix[aman_idx, 0:5])
    night_pct = (night_msgs / heatmap_matrix[aman_idx].sum()) * 100
    assigned_archetypes['Aman'] = ("THE NIGHT OWL", f"({night_pct:.1f}% msgs between 23h-04h)")

    karan_msgs = [m for m in messages if m['sender'] == 'Karan']
    avg_words = np.mean([len(m['text'].split()) for m in karan_msgs])
    assigned_archetypes['Karan'] = ("THE STORYTELLER", f"(avg {avg_words:.1f} words per msg)")

    neha_msgs = [m for m in messages if m['sender'] == 'Neha']
    caps_cnt = sum(1 for m in neha_msgs if (m['text'].isupper() and len(m['text']) > 3) or m['text'].count('!') >= 2)
    caps_pct = (caps_cnt / len(neha_msgs)) * 100
    assigned_archetypes['Neha'] = ("THE DRAMA QUEEN", f"({caps_pct:.1f}% ALL-CAPS messages)")

    ghost_days = silent_streaks['Vikas'][0]
    assigned_archetypes['Vikas'] = ("THE GHOST", f"(silent on {ghost_days} of {total_days} days)")

    return assigned_archetypes
archetype_results = detect_archetypes_exclusive(messages, participants, heatmap_matrix, person_to_idx, silent_streaks, total_days)

print("PERSONALITY ARCHETYPES")
for p in participants:
    title, desc = archetype_results[p]
    print(f"{p:<10} → {title:<22} {desc}")

print("\n" + "="*60)
print("Generated by GroupDNA | Built with Python + NumPy")
print("="*60)

PERSONALITY ARCHETYPES
Aman       → THE NIGHT OWL          (80.4% msgs between 23h-04h)
Karan      → THE STORYTELLER        (avg 57.0 words per msg)
Neha       → THE DRAMA QUEEN        (63.3% ALL-CAPS messages)
Priya      → THE GROUP MOM          (caring keyword score: 621)
Rahul      → THE SPAMMER            (avg 4.5 msgs in a row)
Vikas      → THE GHOST              (silent on 11 of 60 days)

Generated by GroupDNA | Built with Python + NumPy


ReflectionHardest Part: Parsing raw chat lines and handling edge cases (like deleted messages and <Media omitted>) using basic string methods without relying on pandas or regex.  
Key Takeaway: Learning how to process time data with datetime and build a text-based activity heatmap using a simple 2D NumPy array.